In [1]:
import pandas as pd
import numpy as np 
import json
import requests # for NBP API 
from database.db_config import get_engine #for loading engine from db_config

##  starting conversion json file to pandas dataframe 


In [2]:
with open ("../data/raw/raw_json_data",'r',encoding = 'utf-8') as f:
    raw_data = json.load(f)
with open("../data/raw/raw_json_data_IT_only", 'r', encoding='utf-8') as f:
    raw_data_it = json.load(f)

## checking the keys


In [3]:
print(raw_data.keys())

dict_keys(['gb', 'us', 'pl'])


## Creating a new list to work with and creating new column "country"


In [4]:
offers_list = []
for country, offers in raw_data.items():
        for offer in offers:
                offer['country'] = country
                offers_list.append(offer)
            
      # adding another json with mostly IT category offers  
for country, offers in raw_data_it.items():
    for offer in offers:
        offer['country'] = country
        offers_list.append(offer)


## Moving data to pandas dataframe 

In [5]:
df_raw = pd.json_normalize(offers_list)
# removing duplicates
df_raw = df_raw.drop_duplicates(subset=['title', 'company.display_name', 'location.display_name', 'country','salary_min','salary_max'])
cols = ["id",
        "country",
        "title",
        "salary_min",
        "salary_max",
        "contract_time",
        "category.label",
        "created",
        "location.display_name",
        "salary_is_predicted",
        "company.display_name"]

df = df_raw[cols].copy()

In [6]:
df

,id,country,title,salary_min,salary_max,contract_time,category.label,created,location.display_name,salary_is_predicted,company.display_name
0,5651321429,gb,Mobile Vehicle Technician,40170.0,44385.0,full_time,Engineering Jobs,2026-03-02T13:47:19Z,"Kingsway, Derby",0,RAC
1,5615788962,gb,Roadside Technician - Cheltenham,36400.0,60000.0,full_time,Engineering Jobs,2026-02-05T17:38:13Z,"Ampney Crucis, Cirencester",0,RAC
2,5615789084,gb,Roadside Technician - Oxford,36400.0,60000.0,full_time,Engineering Jobs,2026-02-05T17:38:16Z,"Middleton Stoney, Bicester",0,RAC
3,5659620935,gb,Mobile Mechanic,40170.0,44385.0,full_time,Engineering Jobs,2026-03-09T20:55:25Z,"Eastcote, Pinner",0,RAC
4,5651321403,gb,Mobile Vehicle Technician - Northampton,40170.0,44385.0,full_time,Engineering Jobs,2026-03-02T13:47:18Z,"Newnham, Daventry",0,RAC
...,...,...,...,...,...,...,...,...,...,...,...
10868,5678707886,pl,Manager ds wsparcia IT,NaN,NaN,full_time,IT,2026-03-26T02:07:16Z,"Warszawa, mazowieckie",0,Klient justjoin.it
10869,5688102520,pl,Senior IT Recruiter,96000.0,144000.0,full_time,HR i rekrutacja,2026-04-03T03:09:17Z,"Kraków, małopolskie",0,dotLinkers - IT Recruitment
11747,5647744743,pl,IT Support Specialist,NaN,NaN,NaN,IT,2026-02-27T22:47:40Z,"Warszawa, mazowieckie",0,EMERGE IT SUPPORT SPÓŁKA Z OGRANICZONĄ ODPOWIE...
11748,5679718241,pl,IT System Engineer,NaN,NaN,NaN,IT,2026-03-26T22:59:14Z,"Gliwice, śląskie",0,GET IT TOGETHER SPÓŁKA Z OGRANICZONĄ ODPOWIEDZ...


## Cleaning the data

In [7]:
df = df.drop("id",axis=1)

In [8]:
df.columns


Index(['country', 'title', 'salary_min', 'salary_max', 'contract_time',
       'category.label', 'created', 'location.display_name',
       'salary_is_predicted', 'company.display_name'],
      dtype='str')

## Checking for columns with many NaN values


In [9]:
df.isnull().sum()

country                     0
title                       0
salary_min                935
salary_max                940
contract_time            1809
category.label              0
created                     0
location.display_name       0
salary_is_predicted         0
company.display_name        6
dtype: int64

In [10]:
df["contract_time"].head(100)

0      full_time
1      full_time
2      full_time
3      full_time
4      full_time
         ...    
96           NaN
97     part_time
98     full_time
99     part_time
100    full_time
Name: contract_time, Length: 100, dtype: str

## Checking the data type of columns

In [11]:
df.dtypes

country                      str
title                        str
salary_min               float64
salary_max               float64
contract_time                str
category.label               str
created                      str
location.display_name        str
salary_is_predicted          str
company.display_name         str
dtype: object

In [12]:
df["contract_time"] = df["contract_time"].fillna("Not specified")

## Contract type cleaning

In [13]:
df.isnull().sum()

country                    0
title                      0
salary_min               935
salary_max               940
contract_time              0
category.label             0
created                    0
location.display_name      0
salary_is_predicted        0
company.display_name       6
dtype: int64

## Skipping fillna for salary columns to maintain statistical integrity.

In [14]:
df.sample(15)

,country,title,salary_min,salary_max,contract_time,category.label,created,location.display_name,salary_is_predicted,company.display_name
2935,us,"Houseparents, Full-Time - Relocation to Hershe...",75325.59,75325.59,full_time,Healthcare & Nursing Jobs,2025-10-03T01:06:04Z,"Imperial, Imperial County",1,Milton Hershey School
2584,us,Residential Youth Caregiver - Relocation to He...,59812.85,59812.85,full_time,Healthcare & Nursing Jobs,2025-10-03T01:20:00Z,"Moraine, Montgomery County",1,Milton Hershey School
2827,us,RN Emergencty Room,52919.53,52919.53,Not specified,Healthcare & Nursing Jobs,2026-03-21T09:38:47Z,"Okefenokee, Ware County",1,Memorial Satilla Health
3945,pl,Principal Backend Engineer @ Get it together,NaN,NaN,full_time,IT,2026-03-18T02:49:26Z,"Warszawa, mazowieckie",0,Get it together
811,gb,Decorator (Self-Employed),39977.58,39977.58,Not specified,Trade & Construction Jobs,2025-05-16T09:51:16Z,"King's Lynn, Norfolk",1,MyJobQuote
8205,us,IT Business Analyst/HR,143927.60,143927.60,full_time,IT Jobs,2026-02-17T17:57:52Z,"Long Island City, Queens",1,World Wide Technology
8634,us,Lead IT Systems Engineer - Public Sector,170964.39,170964.39,full_time,Engineering Jobs,2026-02-18T18:00:42Z,"The Woodlands, Montgomery County",1,Lumen
8147,us,IT Business Analyst/HR,130722.28,130722.28,full_time,IT Jobs,2026-02-14T17:52:14Z,"Minot, Ward County",1,World Wide Technology
41,gb,Apprentice Educator,24960.00,24960.00,Not specified,Teaching Jobs,2026-03-19T17:55:03Z,"Hatch Warren, Basingstoke",0,Busy Bees
6856,gb,IT Project Engineer,35000.00,45000.00,full_time,IT Jobs,2026-01-27T11:41:26Z,"Charnock Richard, Chorley",0,Bowdon Associates Limited


## Changing columns names to more pleasant and readable


In [15]:
df = df.rename(columns={"category.label":"category","created":"posting_date","company.display_name": "company_name","salary_is_predicted":"is_estimated","location.display_name":"location","contract_time":"employment_type"})

In [16]:
df.columns

Index(['country', 'title', 'salary_min', 'salary_max', 'employment_type',
       'category', 'posting_date', 'location', 'is_estimated', 'company_name'],
      dtype='str')

## Changing to boolean type and combining IT Jobs with IT 

In [17]:
df['category'] = df['category'].replace('IT Jobs','IT')

df["is_estimated"] = df["is_estimated"].astype(int).astype(bool)
df["is_estimated"]

0        False
1        False
2        False
3        False
4        False
         ...  
10868    False
10869    False
11747    False
11748    False
11749    False
Name: is_estimated, Length: 5867, dtype: bool

## Using national polish bank API for new columns with currency exchange

In [18]:
nbp_url = "http://api.nbp.pl/api/exchangerates/tables/a/?format=json"
rates_data = requests.get(nbp_url).json()[0]['rates'] # nbp key in json is 0 
currency_map = {'pl': 1.0}
for r in rates_data:
        if r["code"] == "USD":
                currency_map["us"] = r["mid"]
        elif r["code"] == "GBP":
                currency_map["gb"] = r["mid"]

## Creating new columns exchange_rate, salary_min_pln and salary_max_pln

In [19]:
currency_map
df["exchange_rate"] = df["country"].map(currency_map)


In [20]:
df["country"].value_counts()


country
us    1987
gb    1969
pl    1911
Name: count, dtype: int64

In [21]:
df['salary_min_pln'] = df['salary_min'] * df['exchange_rate']
df['salary_max_pln'] = df['salary_max'] * df['exchange_rate']
df.head(5)

,country,title,salary_min,salary_max,employment_type,category,posting_date,location,is_estimated,company_name,exchange_rate,salary_min_pln,salary_max_pln
0,gb,Mobile Vehicle Technician,40170.0,44385.0,full_time,Engineering Jobs,2026-03-02T13:47:19Z,"Kingsway, Derby",False,RAC,4.9055,197053.935,217730.6175
1,gb,Roadside Technician - Cheltenham,36400.0,60000.0,full_time,Engineering Jobs,2026-02-05T17:38:13Z,"Ampney Crucis, Cirencester",False,RAC,4.9055,178560.200,294330.0000
2,gb,Roadside Technician - Oxford,36400.0,60000.0,full_time,Engineering Jobs,2026-02-05T17:38:16Z,"Middleton Stoney, Bicester",False,RAC,4.9055,178560.200,294330.0000
3,gb,Mobile Mechanic,40170.0,44385.0,full_time,Engineering Jobs,2026-03-09T20:55:25Z,"Eastcote, Pinner",False,RAC,4.9055,197053.935,217730.6175
4,gb,Mobile Vehicle Technician - Northampton,40170.0,44385.0,full_time,Engineering Jobs,2026-03-02T13:47:18Z,"Newnham, Daventry",False,RAC,4.9055,197053.935,217730.6175


# converting 0 values to NaN so they can be ignored on analysis charts

In [22]:
(df["salary_min_pln"] == 0).sum()


np.int64(32)

In [23]:
(df["salary_max_pln"] == 0).sum()


np.int64(0)

In [24]:
df["salary_min_pln"] = df["salary_min_pln"].replace(0,np.nan)
(df["salary_min_pln"] == 0).sum()


np.int64(0)

# average salary in pln value for kde analysis and conversion of monthly salaries to yearly 

In [25]:

df["salary_avg_pln"] = (df["salary_min_pln"] + df["salary_max_pln"]) / 2
is_monthly = df['salary_avg_pln'] < 50000
df.loc[is_monthly, ['salary_min_pln', 'salary_max_pln', 'salary_avg_pln']] *= 12
df['salary_spread_pct'] = ((df['salary_max_pln'] - df['salary_min_pln']) / df['salary_min_pln']) * 100

In [26]:
df["title"].value_counts()

title
Lead IT Systems Engineer - Public Sector                     188
IT Business Analyst/HR                                       151
IT Manager                                                    94
IT Project Manager                                            55
IT Support Engineer                                           54
                                                            ... 
Analityk IT Analityczka IT w Obszarze bankowości mobilnej      1
Specjalista/ka ds. Projektów IT / IT Project Manager           1
IT Team Lead                                                   1
Manager ds wsparcia IT                                         1
Senior IT Recruiter                                            1
Name: count, Length: 2794, dtype: int64

# Feature engineering for seniority analysis (unfortunately better data is required).

In [27]:

def categorize_seniority(title):
    t = str(title).lower()
    if any(x in t for x in ['senior', 'lead', 'principal', 'expert', 'starszy', 'kierownik', 'head']):
        return 'Senior'
    if any(x in t for x in ['junior', 'intern', 'trainee', 'praktyk', 'młodszy', 'staż', 'graduate', 'asystent']):
        return 'Junior'
    if any(x in t for x in ['mid', 'regular', 'intermediate', 'middle']):
        return 'Mid'
    return 'Not Specified'


df.loc[df['category'] == 'IT', 'IT_seniority_level'] = df[df['category'] == 'IT']['title'].apply(categorize_seniority)

# Many roles aren't specified in the current dataset

In [28]:
df['IT_seniority_level'].value_counts()

IT_seniority_level
Not Specified    2571
Senior            722
Junior             93
Mid                 6
Name: count, dtype: int64

# Changing dates to datetime type format

In [29]:
df['posting_date'] = pd.to_datetime(df['posting_date'])
df['day_of_week'] = df['posting_date'].dt.day_name() 



# Changing is_estimated to boolean type 

In [30]:
df['is_estimated'] = df['is_estimated'].astype(bool)
df['is_estimated']

0        False
1        False
2        False
3        False
4        False
         ...  
10868    False
10869    False
11747    False
11748    False
11749    False
Name: is_estimated, Length: 5867, dtype: bool

# Exporting to csv for readability on github

In [31]:
df.to_csv('../data/processed/offers_cleaned.csv', index=False)



# checking counted values in the category column

In [32]:
df["category"].value_counts()

category
IT                                  3392
Healthcare & Nursing Jobs            834
Engineering Jobs                     424
Trade & Construction Jobs            364
Logistics & Warehouse Jobs           138
Accounting & Finance Jobs            119
Teaching Jobs                        115
Hospitality & Catering Jobs           72
Sales Jobs                            47
Part time Jobs                        40
Unknown                               39
Travel Jobs                           38
Retail Jobs                           34
PR, reklama, marketing                24
HR i rekrutacja                       24
PR, Advertising & Marketing Jobs      20
Customer Services Jobs                18
Księgowość i finanse                  18
Inna/ogólna                           17
Social work Jobs                      14
Consultancy Jobs                      13
Manufacturing Jobs                     9
Graduate Jobs                          9
Admin Jobs                             9
Inżynie

## Creating column for wage transparency 

In [33]:
df['has_salary'] = df['salary_min'].notna()

salary_transparency = df.groupby('country')['has_salary'].agg(['count', 'sum', 'mean'])

salary_transparency.columns = ['Total Offers', 'Offers with Salary', 'Transparency Rate']
salary_transparency['Transparency Rate'] = (salary_transparency['Transparency Rate'] * 100).round(2).astype(str) + '%'
salary_transparency


,Total Offers,Offers with Salary,Transparency Rate
country,,,
gb,1969,1969,100.0%
pl,1911,976,51.07%
us,1987,1987,100.0%


# checking the healthcare and nursing jobs in UK 

In [34]:
df[(df["country"] == "gb") & (df["category"] == "Healthcare & Nursing Jobs")]

,country,title,salary_min,salary_max,employment_type,category,posting_date,location,is_estimated,company_name,exchange_rate,salary_min_pln,salary_max_pln,salary_avg_pln,salary_spread_pct,IT_seniority_level,day_of_week,has_salary
25,gb,Pharmacy Manager,47456.85,47456.85,full_time,Healthcare & Nursing Jobs,2026-02-18 14:48:17+00:00,"Great Hallingbury, Bishop's Stortford",True,Ramsay Health Care,4.9055,232799.577675,232799.577675,232799.577675,0.000000,NaN,Wednesday,True
29,gb,Health Affairs Manager UK&I,45891.07,45891.07,full_time,Healthcare & Nursing Jobs,2026-03-24 11:18:26+00:00,"West Camel, Yeovil",True,Crown Pet Foods Ltd,4.9055,225118.643885,225118.643885,225118.643885,0.000000,NaN,Tuesday,True
38,gb,Theatre Scrub Nurse/ODP - Orthopaedics/Spinal,50651.33,50651.33,full_time,Healthcare & Nursing Jobs,2026-03-11 21:17:07+00:00,"Charlton, Andover",True,Ramsay Health Care,4.9055,248470.099315,248470.099315,248470.099315,0.000000,NaN,Wednesday,True
118,gb,Engineering technician,39267.69,39267.69,full_time,Healthcare & Nursing Jobs,2026-04-01 02:01:13+00:00,"Nine Ashes, Ingatestone",True,Ramsay Health Care,4.9055,192627.653295,192627.653295,192627.653295,0.000000,NaN,Wednesday,True
131,gb,Pharmacy Manager,45767.52,45767.52,full_time,Healthcare & Nursing Jobs,2026-02-18 14:48:17+00:00,"Oaklands, St. Albans",True,Ramsay Health Care,4.9055,224512.569360,224512.569360,224512.569360,0.000000,NaN,Wednesday,True
147,gb,Senior Staff Nurse - Wards,44778.06,44778.06,full_time,Healthcare & Nursing Jobs,2025-09-16 20:57:40+00:00,"Crowfield, Brackley",True,Ramsay Health Care,4.9055,219658.773330,219658.773330,219658.773330,0.000000,NaN,Tuesday,True
155,gb,Theatre Scrub Nurse/ODP - Orthopaedics,44793.78,44793.78,full_time,Healthcare & Nursing Jobs,2026-02-01 00:15:48+00:00,"Colchester, Essex",True,Ramsay Health Care,4.9055,219735.887790,219735.887790,219735.887790,0.000000,NaN,Sunday,True
202,gb,High Intensity Team Leader - CBT / EMDR - Wake...,57000.00,62700.00,full_time,Healthcare & Nursing Jobs,2026-03-15 07:45:26+00:00,"Milnthorpe, Wakefield",False,Turning Point,4.9055,279613.500000,307574.850000,293594.175000,10.000000,NaN,Sunday,True
204,gb,Occupational Therapist,41010.99,41010.99,full_time,Healthcare & Nursing Jobs,2026-03-20 17:20:13+00:00,"Wakefield, West Yorkshire",True,Witherslack Group,4.9055,201179.411445,201179.411445,201179.411445,0.000000,NaN,Friday,True
223,gb,Hospital Engineer,37549.12,37549.12,full_time,Healthcare & Nursing Jobs,2026-04-01 02:01:13+00:00,"Rivenhall, Witham",True,Ramsay Health Care,4.9055,184197.208160,184197.208160,184197.208160,0.000000,NaN,Wednesday,True


# checking the amount of IT jobs in Poland 

In [35]:
df[(df["country"]== "pl") & (df["category"] == "IT")]["country"].count()

np.int64(1777)

# checking the duplicates

In [36]:
df.duplicated(subset=['title', 'company_name', 'location', 'country', 'salary_min', 'salary_max']).sum()

np.int64(0)

# counting how many rows are under each country column

In [37]:
df["country"].value_counts()

country
us    1987
gb    1969
pl    1911
Name: count, dtype: int64

## Data quality checks/testing

In [46]:
def run_data_quality_checks(df):
    # columns Check
    assert df['title'].notna().all(), "Found null values in 'title'"
    assert df['country'].notna().all(), "Found null values in 'country'"
    
    # currency conversion check
    if df['salary_min_pln'].notna().any():
        assert (df[df['salary_min_pln'].notna()]['salary_min_pln'] > 0).all(), "Negative/zero values in PLN salary"

    # 3. seniority category check
    valid_labels = {'Senior', 'Junior', 'Mid', 'Not Specified'}
    actual_labels = set(df['IT_seniority_level'].dropna().unique())
    
    assert actual_labels.issubset(valid_labels), f"Unexpected seniority labels: {actual_labels - valid_labels}"

    # duplicate check
    dupes = df.duplicated(subset=['title', 'company_name', 'location', 'country', 'salary_min', 'salary_max']).sum()
    assert dupes == 0, f"Found {dupes} duplicate rows"
    
    # date consistency
    now = pd.Timestamp.now(tz='UTC')
    assert (df['posting_date'] <= now).all(), "Found posting dates in the future"

    return "All checks passed"

In [47]:
try:
    status = run_data_quality_checks(df)
    print(f"PASSED: {status}.")
except AssertionError as e:
    print(f"FAILED: Data quality issue: {e}")

PASSED: All checks passed.


In [41]:
df["IT_seniority_level"].value_counts()

IT_seniority_level
Not Specified    2571
Senior            722
Junior             93
Mid                 6
Name: count, dtype: int64

# SQL export for unnormalized sql table offers 

In [38]:
engine = get_engine()
df.to_sql('offers', con=engine, if_exists='replace', index=False)


5867